# enso1: response of the equatorial ocean to a wind stress patch
pyOM2: py_config/enso1.py

One layer with the equivalent depth c_n^2/g in a closed basin on the equatorial
beta plane. A westerly wind patch acts for three days, Kelvin and Rossby waves
propagate and reflect at the coasts. Sponge layers damp the flow near the
northern and southern walls. As in pyOM2 the wind stress acts on the equivalent
depth, so the response is strongly nonlinear. Reduce taux for a linear response.

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.pe_realistic.model import model

In [ ]:
beta       = 2e-11
Re         = 500e3                  # equatorial Rossby radius
cn         = Re**2*beta             # gravity wave speed
nx, ny, nz = 128, 128, 1
dx, dt     = 50e3, 1800.
y0         = ny*dx/2                # equator
x0         = 1.5e6                  # centre of the wind patch
taux       = 1e-4                   # wind stress
t_wind     = 3*86400.               # duration of the wind
nsteps     = 4800                   # 100 days
plot_every = 96

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=cn**2/9.81, dt=dt,
           coord_degree=False, set_zero_at_surface=True,
           periodic_in_x=False, periodic_in_y=False,
           eq_of_state=0, implicit_free_surface=True, explicit_free_surface=False,
           epsilon=1e-12, max_iterations=5000, superbee_advection=True, Ahbi=0.05e11)

In [ ]:
class enso1(model):

    def set_coriolis(self):
        f = beta*(self.yt - y0)
        self.coriolis_t = OM.np.asarray(f[:, None] + 0*self.xt[None, :])

    def set_initial_conditions(self):
        zeros = np.zeros((nz, ny, nx))
        super().set_initial_conditions(zeros, zeros, zeros, zeros, np.zeros((ny, nx)))
        XU, YT = np.meshgrid(self.xu, self.yt)
        self.taux = OM.np.asarray(taux*np.exp(-(YT - y0)**2/(3*Re)**2)*np.exp(-(XU - x0)**2/0.5e6**2))
        y1, y2, dy = 10*dx, (ny - 10)*dx, 20*dx
        self.sponge = OM.np.asarray(1./(10*3600.)*(np.exp(-(YT - y1)**2/dy**2) + np.exp(-(YT - y2)**2/dy**2)))
        self.time = 0.

    def set_forcing(self, tau):
        self.time += self.dt
        ks = -self.halo - 1
        if self.time <= t_wind:
            self.du = OM.add_to_array(self.du, (tau, ks), self.taux/self.dzt[ks]*self.maskU[ks])
        self.du = OM.add_to_array(self.du, (tau,), -self.sponge*self.u)
        self.dv = OM.add_to_array(self.dv, (tau,), -self.sponge*self.v)

In [ ]:
M = enso1(par)
M.set_coriolis()
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
h = M.halo
hov = []                            # p_s along the equator for the Hovmoeller diagram

In [ ]:
def plot(t, tau):
    clear_output(wait=True)
    p_s = M.p_s[tau, h:-h, h:-h]
    hov.append((t, p_s[ny//2]*1))
    plt.figure(figsize=(7, 6))
    plt.pcolormesh(M.xt_gl/1e3, M.yt_gl/1e3, p_s/9.81, cmap='RdBu_r')
    plt.colorbar(label='$\\eta$ [m]'); plt.xlabel('x [km]'); plt.ylabel('y [km]')
    plt.title('t = %.1f days' % (t/86400)); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt, tau)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
# Hovmoeller diagram along the equator
t = np.array([a[0] for a in hov]); p = np.array([a[1] for a in hov])
plt.figure(figsize=(6, 5))
plt.pcolormesh(M.xt_gl/1e3, t/86400, p/9.81, cmap='RdBu_r')
plt.colorbar(label='$\\eta$ [m]'); plt.xlabel('x [km]'); plt.ylabel('t [days]')
plt.title('c_n = %.1f m/s' % cn); plt.show()